<a href="https://colab.research.google.com/github/Datdeptrai912005/DemoGit/blob/main/MLLMS_Traine_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!pip install transformers datasets evaluate accelerate scikit-learn

In [ ]:
from unsloth import FastLanguageModel
import torch
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-1.5B-Instruct",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
print("✅ Tải mô hình thành công ")

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "chatml",
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

train_dataset = load_dataset("json", data_files="/content/drive/MyDrive/train_data.jsonl", split="train")
test_dataset = load_dataset("json", data_files="/content/drive/MyDrive/test_data.jsonl", split="train")
train_dataset = train_dataset.map(formatting_prompts_func, batched=True)
test_dataset = test_dataset.map(formatting_prompts_func, batched=True)

print(f"📚 Đã nạp {len(train_dataset)} mẫu để Dạy (Train)")
print(f"📝 Đã nạp {len(test_dataset)} mẫu để Kiểm tra (Test)")

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_dataset,
    eval_dataset = test_dataset,
    dataset_text_field = "text",
    max_seq_length = 2048,
    dataset_num_proc = 2,
    packing = False,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        num_train_epochs = 1,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 20,
        eval_strategy = "steps",
        eval_steps = 100,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)
trainer_stats = trainer.train()

In [ ]:

save_path = "/content/drive/MyDrive/MLLA /Qwen_FakeNews_Model"
model.save_pretrained(save_path)
tokenizer.save_pretrained(save_path)
print(f"✅ Đã lưu thành công trí tuệ của mô hình tại: {save_path}")

In [ ]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-multilingual-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)
def preprocess_function(examples):
    texts = []
    labels = []
    for conv in examples["messages"]:
        user_text = next((msg["content"] for msg in conv if msg["role"] == "user"), "")
        label_text = next((msg["content"] for msg in conv if msg["role"] == "assistant"), "0")

        texts.append(user_text)
        labels.append(int(label_text))

    result = tokenizer(texts, padding="max_length", truncation=True, max_length=512)
    result["labels"] = labels
    return result

print("Đang tải dữ liệu từ Drive...")
train_dataset = load_dataset("json", data_files="/content/drive/MyDrive/train_data.jsonl", split="train")
test_dataset = load_dataset("json", data_files="/content/drive/MyDrive/test_data.jsonl", split="train")

tokenized_train = train_dataset.map(preprocess_function, batched=True, remove_columns=["messages"])
tokenized_test = test_dataset.map(preprocess_function, batched=True, remove_columns=["messages"])

print(f"✅ Đã nạp {len(tokenized_train)} mẫu Train và {len(tokenized_test)} mẫu Test cho mBERT.")

In [ ]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

# 1. Định nghĩa cách chấm điểm (Accuracy và F1-Score)
metric_acc = evaluate.load("accuracy")
metric_f1 = evaluate.load("f1")

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    f1 = metric_f1.compute(predictions=predictions, references=labels)["f1"]
    return {"accuracy": acc, "f1": f1}

# 2. Cấu hình thông số học tập
training_args = TrainingArguments(
    output_dir="./mbert_results",
    learning_rate=2e-5,
    per_device_train_batch_size=16, # mBERT nhẹ nên có thể nhồi 16 câu cùng lúc
    per_device_eval_batch_size=16,
    num_train_epochs=3, # mBERT thường cần học 3 vòng mới ngấm
    weight_decay=0.01,
    eval_strategy="epoch", # Chấm điểm sau mỗi vòng học
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True, # Tăng tốc độ train trên GPU T4
)

# 3. Tạo "Giáo viên" huấn luyện
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_test,
    processing_class=tokenizer, # <-- TÔI ĐÃ SỬA DÒNG NÀY CHO BẠN
    compute_metrics=compute_metrics,
)
# 4. BẮT ĐẦU HỌC!
print("Bắt đầu huấn luyện mBERT...")
trainer.train()

# 5. LÀM BÀI THI CUỐI KỲ VÀ IN KẾT QUẢ
print("\n" + "="*40)
print("🏆 KẾT QUẢ CUỐI CÙNG CỦA mBERT TRÊN TẬP TEST 🏆")
print("="*40)
final_results = trainer.evaluate(tokenized_test)
print(f"Độ chính xác (Accuracy): {final_results['eval_accuracy'] * 100:.2f}%")
print(f"Điểm F1-Score:           {final_results['eval_f1'] * 100:.2f}%")
print("="*40)

# (Tùy chọn) Lưu mô hình
trainer.save_model("/content/drive/MyDrive/mBERT_FakeNews_Model")
print("Đã lưu mô hình mBERT lên Drive.")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import numpy as np

print("Đang thu thập đáp án của mBERT...")
predictions = trainer.predict(tokenized_test)
y_pred = np.argmax(predictions.predictions, axis=-1)
y_true = predictions.label_ids

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Tin thật (0)', 'Tin giả (1)'],
            yticklabels=['Tin thật (0)', 'Tin giả (1)'])

plt.title('Sơ đồ Phân tích lỗi mBERT', fontsize=15, fontweight='bold')
plt.xlabel('AI Dự Đoán', fontsize=12)
plt.ylabel('Đấp Án Thực Tế', fontsize=12)
plt.show()

In [ ]:

from unsloth import FastLanguageModel
from datasets import load_dataset
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from tqdm import tqdm
import json

print("Đang tải lại mô hình Qwen từ Drive...")
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "/content/drive/MyDrive/MLLA /Qwen_FakeNews_Model",
    max_seq_length = 2048,
    dtype = None,
    load_in_4bit = True,
)
FastLanguageModel.for_inference(model)

test_dataset = load_dataset("json", data_files="/content/drive/MyDrive/test_data.jsonl", split="train")

y_true = []
y_pred = []

print("Đang yêu cầu Qwen làm toàn bộ đề thi Test (Sẽ mất vài phút)...")
for item in tqdm(test_dataset):
    # Lấy đáp án thật
    true_label = next(msg["content"] for msg in item["messages"] if msg["role"] == "assistant")
    y_true.append(int(true_label))

    # Cho Qwen đọc và đoán
    user_text = next(msg["content"] for msg in item["messages"] if msg["role"] == "user")
    messages = [
        {"role": "system", "content": "Phân tích tin giả."},
        {"role": "user", "content": user_text}
    ]

    inputs = tokenizer.apply_chat_template(messages, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
    outputs = model.generate(input_ids=inputs, max_new_tokens=10, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    response = tokenizer.batch_decode(outputs)[0]

    pred_str = response.split("<|im_start|>assistant\n")[-1].replace("<|im_end|>", "").strip()

    if "1" in pred_str:
        y_pred.append(1)
    else:
        y_pred.append(0)

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Oranges',
            xticklabels=['Tin thật (0)', 'Tin giả (1)'],
            yticklabels=['Tin thật (0)', 'Tin giả (1)'])

plt.title('Sơ đồ Phân tích lỗi (Confusion Matrix) - Qwen2.5', fontsize=15, fontweight='bold')
plt.xlabel('AI Dự Đoán', fontsize=12)
plt.ylabel('Đáp Án Thực Tế', fontsize=12)
plt.show()